In [1]:
from mistralai.azure.client import MistralAzure
import pandas as pd

df = pd.read_csv("datasets/product_vendor_combinations.csv")
df



,product_name,vendor_name
0,freescout,freescout-help-desk
1,civetweb,CivetWeb
2,CPython,Python Software Foundation
3,GoAnywhere MFT,Fortra
4,Firefox,Mozilla
...,...,...
43815,haystack,deepset-ai
43816,Establishment Billing Management System,SourceCodester
43817,eWeLink Cloud Service,CoolKIt
43818,WANotifier,Unknown


In [5]:
df = df.iloc[1000:43820]
df

,product_name,vendor_name
1000,Form Maker by 10Web,Unknown
1001,EMUI,Huawei
1002,TOSRFEC.SYS,Dynabook Inc.
1003,Cart,VirtueMart
1004,Net::CIDR::Lite,STIGTSP
...,...,...
43815,haystack,deepset-ai
43816,Establishment Billing Management System,SourceCodester
43817,eWeLink Cloud Service,CoolKIt
43818,WANotifier,Unknown


In [ ]:

endpoint = #https here
api_key = #password here

In [40]:
import time
import logging
from pathlib import Path

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger(__name__)

client = MistralAzure(server_url=endpoint, api_key=api_key)

OUTPUT_FILE = Path("response ai.txt")
BATCH_SIZE = 10
MAX_RETRIES = 2
RETRY_DELAY = 5   
BATCH_DELAY = 1  


def api_call(prompt_lines):
    resp = client.chat.complete(
        model="Mistral-Large-3",
        messages=[
            {
                "role": "system",
                "content": (
                    "Use the vendor name and product name to identify the 5 most fitting generic product categories, ranked from most to least specific. "
                    "Respond ONLY with a valid JSON array. Each object must have exactly these keys: "
                    "'Original Vendor', 'Original Product', 'Category_1', 'Category_2', 'Category_3', 'Category_4', 'Category_5'. "
                    "Do not include any extra text, markdown, or explanations outside the JSON array."
                ),
            },
            {
                "role": "user",
                "content": (
                    f"The items to categorize are:\n{prompt_lines}"
                ),
            },
        ],
    )
    return resp.choices[0].message.content

def save_response(response_text, batch_index):
    with OUTPUT_FILE.open("a", encoding="utf-8") as f:
        f.write(f"\n--- Batch starting at row {batch_index} ---\n")
        f.write(response_text)
        f.write("\n")


def api_call_with_retry(prompt_lines, batch_index, expected_count):
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = api_call(prompt_lines)
            return response
        except Exception as e:
            logger.warning(f"Batch {batch_index} attempt {attempt}: API error — {e}")

        if attempt < MAX_RETRIES:
            logger.info(f"Retrying in {RETRY_DELAY}s...")
            time.sleep(RETRY_DELAY)

    logger.error(f"Batch {batch_index}: all {MAX_RETRIES} attempts failed. Skipping.")
    return None


total_rows = len(df)
successful_batches = 0
failed_batches = []

logger.info(f"Starting processing of {total_rows} rows in batches of {BATCH_SIZE}.")

for i in range(0, total_rows, BATCH_SIZE):
    batch = df.iloc[i : i + BATCH_SIZE]

    prompt_data = "\n".join(
        f"Vendor: {row['vendor_name']}, Product: {row['product_name']}"
        for _, row in batch.iterrows()
    )

    logger.info(f"Processing rows {i}–{min(i + BATCH_SIZE, total_rows) - 1} / {total_rows - 1}...")

    response = api_call_with_retry(prompt_data, batch_index=i, expected_count=len(batch))

    if response:
        save_response(response, batch_index=i)
        successful_batches += 1
    else:
        failed_batches.append(i)

    time.sleep(BATCH_DELAY)


logger.info(f"Done. {successful_batches} batches saved to '{OUTPUT_FILE}'.")
if failed_batches:
    logger.warning(f"Failed batch start indices (rows): {failed_batches}")

2026-06-16 15:56:44,558 - INFO - Starting processing of 42820 rows in batches of 10.
2026-06-16 15:56:44,560 - INFO - Processing rows 0–9 / 42819...
2026-06-16 15:56:54,984 - INFO - HTTP Request: POST https://ace-studentem.services.ai.azure.com/models/chat/completions?api-version=2024-05-01-preview "HTTP/1.1 200 OK"
2026-06-16 15:56:55,995 - INFO - Processing rows 10–19 / 42819...
2026-06-16 15:57:06,515 - INFO - HTTP Request: POST https://ace-studentem.services.ai.azure.com/models/chat/completions?api-version=2024-05-01-preview "HTTP/1.1 200 OK"
2026-06-16 15:57:07,521 - INFO - Processing rows 20–29 / 42819...
2026-06-16 15:57:21,192 - INFO - HTTP Request: POST https://ace-studentem.services.ai.azure.com/models/chat/completions?api-version=2024-05-01-preview "HTTP/1.1 200 OK"
2026-06-16 15:57:22,198 - INFO - Processing rows 30–39 / 42819...
2026-06-16 15:57:37,226 - INFO - HTTP Request: POST https://ace-studentem.services.ai.azure.com/models/chat/completions?api-version=2024-05-01-pre

In [3]:
df_final = pd.read_csv("datasets/5 categories.csv")

In [24]:
#2nd time
import time
import logging
from pathlib import Path

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger(__name__)

client = MistralAzure(server_url=endpoint, api_key=api_key)

OUTPUT_FILE = Path("response ai.txt")
BATCH_SIZE = 20
MAX_RETRIES = 2
RETRY_DELAY = 5   
BATCH_DELAY = 1  
CATEGORIES = [
    "WordPress / WooCommerce Plugins & Themes",
    "Unclassified / Other",
    "General Software Applications",
    "General CMS Platforms (Non-WordPress)",
    "Networking Hardware (Routers, Switches, Access Points)",
    "Enterprise Software (General Business Applications)",
    "Software Development Libraries & SDKs",
    "Hardware & Electronics (General/Unclassified)",
    "ERP Software",
    "System-on-Chip, Processors & Semiconductor Components",
    "Project & Task Management Software",
    "Operating Systems",
    "CI/CD & DevOps Automation Tools",
    "Identity & Access Management / Authentication Software",
    "Security Information & Vulnerability Management Software",
    "Web Server Software",
    "Industrial Control Systems & PLCs",
    "Web Applications (General/Unclassified)",
    "Monitoring, Logging & Observability Software",
    "Endpoint Security & Antivirus Software",
    "SEO & Digital Marketing Tools",
    "Video Conferencing & Collaboration Software",
    "Data Storage & Backup Software",
    "Artificial Intelligence & Machine Learning Software",
    "Learning Management & Educational Software",
    "Statistical & Data Analysis Software",
    "Productivity & Office Software",
    "E-Commerce Platforms (Standalone)",
    "Network Security & Firewall Software",
    "IT Asset & Endpoint Management Software",
    "IoT Devices & Embedded Systems",
    "API & Integration Tools",
    "Container & Virtualization Platforms",
    "Building Management & Automation Systems",
    "CRM Software",
    "IT Infrastructure & Server Management Software",
    "Video Surveillance & Security Camera Systems",
    "ETL, Data Integration & Business Intelligence Software",
    "System Administration & Scripting Tools",
    "Cloud Infrastructure & Orchestration Software",
    "Image, Photo & Video Editing Software",
    "Database Management Software",
    "Document Management Software",
    "Media Players & Streaming Client Software",
    "Wiki & Knowledge Base Software",
    "FTP & Managed File Transfer Software",
    "Application Server / Middleware",
    "Payment Processing & FinTech Software",
    "Point of Sale (POS) & Retail Management Software",
    "Industrial Automation Hardware & Equipment",
    "Booking & Reservation Systems (Non-WordPress)",
    "Mobile Applications (Android/iOS)",
    "Integrated Development Environments (IDEs) & Code Editors",
    "VoIP & Telephony Systems",
    "Customer Support & Help Desk Software",
    "Cryptography, Encryption & PKI Software",
    "Web Development Frameworks & Frontend Build Tools",
    "Web Browsers & Extensions",
    "Manufacturing Execution Systems & Engineering Software",
    "Inventory & Supply Chain Management Software",
    "VPN & Remote Access Software",
    "Wearable & Consumer Electronics",
    "Restaurant, Hospitality & Food Service Software",
    "Accounting & Financial Management Software",
    "Messaging & Chat Applications",
    "Version Control & Source Code Management Tools",
    "Programming Language Runtimes & Interpreters",
    "Power, Energy & Electrical Equipment",
    "Electronic Health Record & Healthcare Management Software",
    "Firmware (General/Unspecified Device)",
    "Human Resources & Payroll Software",
    "PDF & Document Viewer/Editor Software",
    "Email & Messaging Server Software",
    "Printers & Imaging Hardware",
    "Physical Access Control & Security Hardware",
    "Mobile Device Hardware & Firmware",
    "Networking & HTTP Client Libraries",
    "DNS, Proxy & Network Privacy Tools",
    "Reverse Proxy & Load Balancer Software",
    "Medical Devices & Diagnostic Equipment",
    "Telecommunications & Network Management Software",
    "Legal, Compliance & Privacy Management Software",
    "Event Management & Ticketing Software",
    "Web Forums & Discussion Platforms",
    "Website Builders & Templates",
    "Email Security & Anti-Spam Software",
    "Penetration Testing & Security Assessment Tools",
    "Blockchain & Cryptocurrency Software",
    "Package Managers & Dependency Management Tools",
    "Data Serialization, Parsing & Encoding Libraries",
    "Social Media Management & Integration Tools",
    "Real Estate & Property Management Software",
    "Network Attached Storage (NAS) & Storage Appliances",
    "AI Writing, Content & Code Assistant Tools",
    "Gaming Software & Game Engines",
    "Web Scraping & Data Extraction Tools",
    "Automotive & Vehicle Systems",
    "Government, Nonprofit & Public Sector Systems",
    "Geospatial, Mapping & GIS Software",
    "Secure Shell & Remote Administration Tools",
    "Laboratory & Scientific Instrumentation",
]


def api_call(prompt_lines):
    resp = client.chat.complete(
        model="Mistral-Large-3",
        messages=[
            {
                "role": "system",
                "content": (
                    "Use the vendor name, product name and categories, ranked from most to least specific, "
                    "to identify the most fitting generic product category. "
                    "Respond ONLY with a valid JSON array. Each object must have exactly these keys: "
                    "'Original Vendor', 'Original Product', 'Generic Category'. "
                    "The categories to choose from are: " + ", ".join(CATEGORIES) + "."
                )
            },
            {
                "role": "user",
                "content": (
                    f"The items to categorize are:\n{prompt_lines}"
                ),
            },
        ],
    )
    return resp.choices[0].message.content

def save_response(response_text, batch_index):
    with OUTPUT_FILE.open("a", encoding="utf-8") as f:
        f.write(f"\n--- Batch starting at row {batch_index} ---\n")
        f.write(response_text)
        f.write("\n")


def api_call_with_retry(prompt_lines, batch_index, expected_count):
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = api_call(prompt_lines)
            return response
        except Exception as e:
            logger.warning(f"Batch {batch_index} attempt {attempt}: API error — {e}")

        if attempt < MAX_RETRIES:
            logger.info(f"Retrying in {RETRY_DELAY}s...")
            time.sleep(RETRY_DELAY)

    logger.error(f"Batch {batch_index}: all {MAX_RETRIES} attempts failed. Skipping.")
    return None


total_rows = len(df_final)
successful_batches = 0
failed_batches = []

logger.info(f"Starting processing of {total_rows} rows in batches of {BATCH_SIZE}.")

for i in range(0, total_rows, BATCH_SIZE):
    batch = df_final.iloc[i : i + BATCH_SIZE]

    prompt_data = "\n".join(
        f"Vendor: {row['Original Vendor']}, Product: {row['Original Product']}, Category_1: {row['Category_1']}, Category_2: {row['Category_2']}, Category_3: {row['Category_3']}, Category_4: {row['Category_4']}, Category_5: {row['Category_5']}"
        for _, row in batch.iterrows()
    )

    logger.info(f"Processing rows {i}–{min(i + BATCH_SIZE, total_rows) - 1} / {total_rows - 1}...")

    response = api_call_with_retry(prompt_data, batch_index=i, expected_count=len(batch))

    if response:
        save_response(response, batch_index=i)
        successful_batches += 1
    else:
        failed_batches.append(i)

    time.sleep(BATCH_DELAY)


logger.info(f"Done. {successful_batches} batches saved to '{OUTPUT_FILE}'.")
if failed_batches:
    logger.warning(f"Failed batch start indices (rows): {failed_batches}")

2026-06-19 11:20:11,770 - INFO - Starting processing of 42771 rows in batches of 20.
2026-06-19 11:20:11,770 - INFO - Processing rows 0–19 / 42770...
2026-06-19 11:20:30,461 - INFO - HTTP Request: POST https://ace-studentem.services.ai.azure.com/models/chat/completions?api-version=2024-05-01-preview "HTTP/1.1 200 OK"
2026-06-19 11:20:31,468 - INFO - Processing rows 20–39 / 42770...
2026-06-19 11:20:47,372 - INFO - HTTP Request: POST https://ace-studentem.services.ai.azure.com/models/chat/completions?api-version=2024-05-01-preview "HTTP/1.1 200 OK"
2026-06-19 11:20:48,378 - INFO - Processing rows 40–59 / 42770...
2026-06-19 11:21:08,805 - INFO - HTTP Request: POST https://ace-studentem.services.ai.azure.com/models/chat/completions?api-version=2024-05-01-preview "HTTP/1.1 200 OK"
2026-06-19 11:21:09,816 - INFO - Processing rows 60–79 / 42770...
2026-06-19 11:21:33,950 - INFO - HTTP Request: POST https://ace-studentem.services.ai.azure.com/models/chat/completions?api-version=2024-05-01-pr

In [4]:
#3rd time
import time
import logging
from pathlib import Path

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger(__name__)

client = MistralAzure(server_url=endpoint, api_key=api_key)

OUTPUT_FILE = Path("datasets/third ai run.txt")
BATCH_SIZE = 20
MAX_RETRIES = 2
RETRY_DELAY = 5   
BATCH_DELAY = 1  
CATEGORIES = [
    "Vehicles",
    "Machinery",
    "Recreational vehicles",
    "Medical equipment",
    "Gas appliances",
    "Toys",
    "Household appliances",
    "Measuring and weighing equipment (MID)",
    "Measuring equipment (non-MID)",
    "Alarm and fire detection systems",
    "Telephones, smartphones, e-readers, wearables",
    "Televisions, monitors, projectors, LED screens and information kiosks",
    "Audio equipment (players, speakers, headphones, microphones)",
    "Computers, laptops, servers",
    "Telecommunications peripheral equipment, switches, routers, wireless access points, media converters, modems, network firewalls and security equipment",
    "Input/output devices (keyboard, mouse, monitors)",
    "Printers, copiers, scanners",
    "Transformers, power supplies, chargers",
    "Batteries",
    "Power electronics",
    "Energy storage systems",
    "Electric motors and generators",
    "Electronic components",
    "Electrical wires and cables",
    "Switches, plugs, sockets, connectors, switchgear",
    "Communication equipment / transmitting-receiving equipment",
    "Smart devices / home automation",
    "Payment and ticketing machines (i.e. self-service terminals – SSTs)",
    "Electrical devices for personal and recreational use",
    "Other category"
]


def api_call(prompt_lines):
    resp = client.chat.complete(
        model="Mistral-Large-3",
        messages=[
            {
                "role": "system",
                "content": (
                    "Use the vendor name, product name and categories, ranked from most to least specific, "
                    "to identify the most fitting generic product category. "
                    "Respond ONLY with a valid JSON array. Each object must have exactly these keys: "
                    "'Original Vendor', 'Original Product', 'Generic Category'. "
                    "If it does not fit any of the categories, choose 'Other category'."
                    "The categories to choose from are: " + ", ".join(CATEGORIES) + "."
                )
            },
            {
                "role": "user",
                "content": (
                    f"The items to categorize are:\n{prompt_lines}"
                ),
            },
        ],
    )
    return resp.choices[0].message.content

def save_response(response_text, batch_index):
    with OUTPUT_FILE.open("a", encoding="utf-8") as f:
        f.write(f"\n--- Batch starting at row {batch_index} ---\n")
        f.write(response_text)
        f.write("\n")


def api_call_with_retry(prompt_lines, batch_index, expected_count):
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = api_call(prompt_lines)
            return response
        except Exception as e:
            logger.warning(f"Batch {batch_index} attempt {attempt}: API error — {e}")

        if attempt < MAX_RETRIES:
            logger.info(f"Retrying in {RETRY_DELAY}s...")
            time.sleep(RETRY_DELAY)

    logger.error(f"Batch {batch_index}: all {MAX_RETRIES} attempts failed. Skipping.")
    return None


total_rows = len(df_final)
successful_batches = 0
failed_batches = []

logger.info(f"Starting processing of {total_rows} rows in batches of {BATCH_SIZE}.")

for i in range(0, total_rows, BATCH_SIZE):
    batch = df_final.iloc[i : i + BATCH_SIZE]

    prompt_data = "\n".join(
        f"Vendor: {row['Original Vendor']}, Product: {row['Original Product']}, Category_1: {row['Category_1']}, Category_2: {row['Category_2']}, Category_3: {row['Category_3']}, Category_4: {row['Category_4']}, Category_5: {row['Category_5']}"
        for _, row in batch.iterrows()
    )

    logger.info(f"Processing rows {i}–{min(i + BATCH_SIZE, total_rows) - 1} / {total_rows - 1}...")

    response = api_call_with_retry(prompt_data, batch_index=i, expected_count=len(batch))

    if response:
        save_response(response, batch_index=i)
        successful_batches += 1
    else:
        failed_batches.append(i)

    time.sleep(BATCH_DELAY)


logger.info(f"Done. {successful_batches} batches saved to '{OUTPUT_FILE}'.")
if failed_batches:
    logger.warning(f"Failed batch start indices (rows): {failed_batches}")

2026-06-23 22:42:04,534 - INFO - Starting processing of 42700 rows in batches of 20.
2026-06-23 22:42:04,534 - INFO - Processing rows 0–19 / 42699...
2026-06-23 22:42:13,082 - INFO - HTTP Request: POST https://ace-studentem.services.ai.azure.com/models/chat/completions?api-version=2024-05-01-preview "HTTP/1.1 200 OK"
2026-06-23 22:42:14,094 - INFO - Processing rows 20–39 / 42699...
2026-06-23 22:42:21,366 - INFO - HTTP Request: POST https://ace-studentem.services.ai.azure.com/models/chat/completions?api-version=2024-05-01-preview "HTTP/1.1 200 OK"
2026-06-23 22:42:22,374 - INFO - Processing rows 40–59 / 42699...
2026-06-23 22:42:28,104 - INFO - HTTP Request: POST https://ace-studentem.services.ai.azure.com/models/chat/completions?api-version=2024-05-01-preview "HTTP/1.1 200 OK"
2026-06-23 22:42:29,111 - INFO - Processing rows 60–79 / 42699...
2026-06-23 22:42:33,541 - INFO - HTTP Request: POST https://ace-studentem.services.ai.azure.com/models/chat/completions?api-version=2024-05-01-pr